In [ ]:
!pip install lightgbm xgboost catboost --quiet

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import os
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
# Load Data

train_path = "train.csv"
test_path = "test.csv"

assert os.path.exists(train_path), "train.csv not found in working directory"
assert os.path.exists(test_path), "test.csv not found in working directory"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# Column names sanity check
print("Columns:", list(train.columns))

# Target and ID columns (adjust here if your names differ)
ID_COL = "id"
TARGET_COL = "smoking"

Train shape: (15000, 24)
Test shape: (10000, 23)
Columns: ['id', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries', 'smoking']


**Feature Engineering**

In [ ]:
def add_features(df):
    """
    Add engineered features to df in-place.
    Works for both train and test. Only creates features if source columns exist.
    """
    cols = df.columns

    # BMI = weight(kg) / (height(m))^2
    if "height(cm)" in cols and "weight(kg)" in cols:
        height_m = df["height(cm)"] / 100.0
        df["BMI"] = df["weight(kg)"] / (height_m ** 2 + 1e-6)

    # Triglyceride / HDL ratio
    if "triglyceride" in cols and "HDL" in cols:
        df["TG_to_HDL"] = df["triglyceride"] / (df["HDL"] + 1e-6)

    # Pulse pressure and mean arterial pressure
    if "systolic" in cols and "relaxation" in cols:
        df["pulse_pressure"] = df["systolic"] - df["relaxation"]
        df["mean_arterial_pressure"] = df["relaxation"] + df["pulse_pressure"] / 3.0

    # Liver enzyme composite features
    has_AST = "AST" in cols
    has_ALT = "ALT" in cols
    has_Gtp = "Gtp" in cols

    if has_AST and has_ALT:
        df["AST_ALT_ratio"] = df["AST"] / (df["ALT"] + 1e-6)
    if has_AST and has_ALT and has_Gtp:
        df["liver_enzyme_sum"] = df["AST"] + df["ALT"] + df["Gtp"]

    # Proteinuria flag from Urine protein (any value > 1 considered abnormal)
    if "Urine protein" in cols:
        df["proteinuria_flag"] = (df["Urine protein"] > 1).astype(int)

    # Any hearing issue (either ear)
    if "hearing(left)" in cols and "hearing(right)" in cols:
        df["any_hearing_issue"] = ((df["hearing(left)"] > 1) | (df["hearing(right)"] > 1)).astype(int)

    # Log transforms for skewed labs (if they exist)
    skewed = [
        "triglyceride",
        "Gtp",
        "ALT",
        "AST",
        "serum creatinine",
        "fasting blood sugar"
    ]
    for col in skewed:
        if col in cols:
            df[col + "_log"] = np.log1p(df[col].clip(lower=0))

    return df

# Apply feature engineering to both train and test
train = add_features(train)
test = add_features(test)

print("Train columns after feature engineering:", len(train.columns))
print("Test columns after feature engineering:", len(test.columns))

Train columns after feature engineering: 38
Test columns after feature engineering: 37


Handling outliers

In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    if col in [ID_COL, TARGET_COL]:
        continue
    # Compute quantiles on train only
    low = train[col].quantile(0.01)
    high = train[col].quantile(0.99)
    train[col] = train[col].clip(low, high)
    if col in test.columns:
        test[col] = test[col].clip(low, high)

In [ ]:
feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET_COL]]
feature_cols_test = [c for c in test.columns if c != ID_COL]

# Make sure both sets of features are aligned
feature_cols = [c for c in feature_cols if c in feature_cols_test]
feature_cols_test = feature_cols  # enforce same order

print("Using", len(feature_cols), "feature columns")

X = train[feature_cols].values
y = train[TARGET_COL].values
X_test = test[feature_cols].values

Using 36 feature columns


In [ ]:
lgb_params = {
    "n_estimators": 2000,
    "learning_rate": 0.02,
    "num_leaves": 31,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "min_child_samples": 20,
    "objective": "binary",
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}

xgb_params = {
    "n_estimators": 2000,
    "learning_rate": 0.02,
    "max_depth": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "min_child_weight": 1,
    "gamma": 0.0,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "tree_method": "hist"  # faster on CPU; change to "gpu_hist" if you enable GPU
}

cat_params = {
    "iterations": 2000,
    "learning_rate": 0.03,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_state": RANDOM_STATE,
    "l2_leaf_reg": 3.0,
    # no bootstrap_type / subsample to avoid Bayesian-weights issue
    "od_type": "Iter",
    "od_wait": 100,
    "verbose": False
}


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))

test_pred_lgb = np.zeros(len(test))
test_pred_xgb = np.zeros(len(test))
test_pred_cat = np.zeros(len(test))

In [ ]:
for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== Fold {fold} ==========")
    X_tr, X_val = X[trn_idx], X[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    # LightGBM
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    test_pred_lgb += lgb_model.predict_proba(X_test)[:, 1] / skf.n_splits
    print("  LGBM fold AUC:", roc_auc_score(y_val, oof_lgb[val_idx]))

    # XGBoost
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_pred_xgb += xgb_model.predict_proba(X_test)[:, 1] / skf.n_splits
    print("  XGB  fold AUC:", roc_auc_score(y_val, oof_xgb[val_idx]))


    # CatBoost
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    test_pred_cat += cat_model.predict_proba(X_test)[:, 1] / skf.n_splits
    print("  CAT  fold AUC:", roc_auc_score(y_val, oof_cat[val_idx]))


========== Fold 1 ==========
[LightGBM] [Info] Number of positive: 4426, number of negative: 7574
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.039086 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2451
[LightGBM] [Info] Number of data points in the train set: 12000, number of used features: 33
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.368833 -> initscore=-0.537225
[LightGBM] [Info] Start training from score -0.537225
  LGBM fold AUC: 0.8981245620031659
  XGB  fold AUC: 0.8970896005468874
  CAT  fold AUC: 0.8981197881957108

========== Fold 2 ==========
[LightGBM] [Info] Number of positive: 4426, number of negative: 7574
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004257 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2439
[LightGBM] 

In [ ]:
auc_lgb = roc_auc_score(y, oof_lgb)
auc_xgb = roc_auc_score(y, oof_xgb)
auc_cat = roc_auc_score(y, oof_cat)

print("\n========== OOF AUCs ==========")
print(f"LGBM OOF AUC: {auc_lgb:.6f}")
print(f"XGB  OOF AUC: {auc_xgb:.6f}")
print(f"CAT  OOF AUC: {auc_cat:.6f}")

# Simple average ensemble AUC (on train)
avg_oof = (oof_lgb + oof_xgb + oof_cat) / 3.0
auc_avg = roc_auc_score(y, avg_oof)
print(f"Simple average ensemble OOF AUC: {auc_avg:.6f}")


========== OOF AUCs ==========
LGBM OOF AUC: 0.888303
XGB  OOF AUC: 0.886644
CAT  OOF AUC: 0.888205
Simple average ensemble OOF AUC: 0.889901


In [ ]:
stack_train = np.vstack([oof_lgb, oof_xgb, oof_cat]).T
stack_test = np.vstack([test_pred_lgb, test_pred_xgb, test_pred_cat]).T

meta = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=RANDOM_STATE
)
meta.fit(stack_train, y)

oof_stack = meta.predict_proba(stack_train)[:, 1]
auc_stack = roc_auc_score(y, oof_stack)
print(f"\nStacked model OOF AUC: {auc_stack:.6f}")

# Final test predictions from stacker
test_pred_stack = meta.predict_proba(stack_test)[:, 1]


Stacked model OOF AUC: 0.889789


In [ ]:
if auc_stack >= auc_avg:
    print("Using stacked predictions for submission.")
    final_pred = test_pred_stack
else:
    print("Using simple average ensemble for submission.")
    final_pred = (test_pred_lgb + test_pred_xgb + test_pred_cat) / 3.0

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET_COL: final_pred
})

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"\nSaved submission file to: {submission_path}")
print(submission.head())

Using simple average ensemble for submission.

Saved submission file to: submission.csv
      id   smoking
0  15000  0.017705
1  15001  0.012632
2  15002  0.342411
3  15003  0.013578
4  15004  0.012512
